# 05 — Quantum Error Mitigation Analysis

This notebook compares **raw IBM hardware execution**, **readout-error mitigation**, and **zero-noise extrapolation (ZNE)** for the benchmark suite.

The objective is to quantify mitigation benefit and experimental overhead before implementing the **adaptive QEM selector** in the next stage.

## Analysis Flow

**Raw hardware data → reference/ideal distribution → Readout mitigation → ZNE → fidelity/success/error metrics → improvement and overhead analysis**

The analysis is deliberately separated from the adaptive decision logic so that the selector can later be evaluated against these measured results without circular reasoning.

In [ ]:
from pathlib import Path
import sys
import json
import math
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

HARDWARE_DIR = PROJECT_ROOT / "data" / "hardware"
MITIGATED_DIR = PROJECT_ROOT / "data" / "mitigated"
IDEAL_DIR = PROJECT_ROOT / "data" / "ideal"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

for directory in [MITIGATED_DIR, RESULTS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BACKEND_NAME = "ibm_kingston"
SHOTS = 4096

print("Project root:", PROJECT_ROOT)
print("Hardware data:", HARDWARE_DIR)
print("Mitigated data:", MITIGATED_DIR)

## 1. Load Hardware and Ideal Data

The notebook accepts previously collected hardware counts. If the hardware benchmark notebook has not yet produced data, the analysis stops with a clear message rather than generating synthetic hardware results.

In [ ]:
hardware_results_path = HARDWARE_DIR / f"{BACKEND_NAME}_hardware_results.csv"
hardware_counts_path = HARDWARE_DIR / f"{BACKEND_NAME}_raw_counts.json"
ideal_counts_path = IDEAL_DIR / "ideal_counts.json"

if not hardware_results_path.exists() or not hardware_counts_path.exists():
    raise FileNotFoundError(
        "Hardware results are missing. Run 04_hardware_benchmarks.ipynb "
        "after completing IBM hardware execution."
    )

hardware_results_df = pd.read_csv(hardware_results_path)
hardware_counts = json.loads(hardware_counts_path.read_text(encoding="utf-8"))

ideal_counts = {}
if ideal_counts_path.exists():
    ideal_counts = json.loads(ideal_counts_path.read_text(encoding="utf-8"))

print("Hardware records:", len(hardware_results_df))
print("Hardware circuits:", list(hardware_counts.keys()))
print("Ideal reference circuits:", len(ideal_counts))

## 2. Distribution Utilities

For probability-distribution comparisons, the ideal simulator output is treated as the reference distribution. The functions below calculate total-variation distance and a normalized classical distribution fidelity.

In [ ]:
def normalize_counts(counts):
    total = sum(counts.values())
    if total == 0:
        return {}
    return {state: value / total for state, value in counts.items()}

def all_states(*count_dicts):
    states = set()
    for counts in count_dicts:
        states.update(counts.keys())
    return sorted(states)

def total_variation_distance(reference_counts, measured_counts):
    p = normalize_counts(reference_counts)
    q = normalize_counts(measured_counts)
    states = all_states(reference_counts, measured_counts)
    return 0.5 * sum(abs(p.get(s, 0.0) - q.get(s, 0.0)) for s in states)

def classical_fidelity(reference_counts, measured_counts):
    p = normalize_counts(reference_counts)
    q = normalize_counts(measured_counts)
    states = all_states(reference_counts, measured_counts)
    return sum(
        math.sqrt(p.get(s, 0.0) * q.get(s, 0.0))
        for s in states
    ) ** 2

def distribution_metrics(reference_counts, measured_counts):
    tvd = total_variation_distance(reference_counts, measured_counts)
    fidelity = classical_fidelity(reference_counts, measured_counts)
    return {
        "TVD": tvd,
        "distribution_fidelity": fidelity,
    }

## 3. Benchmark Success Metrics

For deterministic benchmarks, success probability is calculated using the same target definitions used in the ideal and noisy stages. For QFT, QPE, QAOA, and QRNG, distribution-level metrics are more appropriate and are retained for later analysis.

In [ ]:
def probability_of_states(counts, states):
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return sum(counts.get(state, 0) for state in states) / total

def benchmark_success(name, counts):
    if name == "Bell_Phi_Plus":
        return probability_of_states(counts, ["00", "11"])

    if name.startswith("GHZ_"):
        n = int(name.split("_")[1])
        return probability_of_states(counts, ["0" * n, "1" * n])

    if name.startswith("Superdense_"):
        message = name.split("_")[1]
        return probability_of_states(counts, [message])

    if name == "Grover_2Q":
        return probability_of_states(counts, ["11"])

    return float("nan")

## 4. Readout-Error Mitigation

A calibration matrix is estimated from the measured single-qubit assignment error when available. For an n-qubit measurement, the tensor product of the individual assignment matrices forms the full measurement calibration matrix.

The inverse/pseudoinverse is then applied to the observed probability vector and negative numerical values are clipped before renormalization.

In [ ]:
def single_qubit_assignment_matrix(p01, p10):
    # Rows: measured 0/1; columns: prepared/ideal 0/1.
    return np.array([
        [1.0 - p01, p10],
        [p01, 1.0 - p10],
    ], dtype=float)

def get_readout_parameters(calibration_df, qubit):
    row = calibration_df[calibration_df["qubit"] == qubit]
    if row.empty:
        return None, None

    row = row.iloc[0]
    p01 = row.get("prob_meas1_prep0")
    p10 = row.get("prob_meas0_prep1")

    if pd.isna(p01) or pd.isna(p10):
        return None, None

    return float(p01), float(p10)

def build_measurement_matrix(num_qubits, calibration_df=None):
    matrix = np.array([[1.0]])

    for q in range(num_qubits):
        if calibration_df is None:
            p01, p10 = 0.0, 0.0
        else:
            p01, p10 = get_readout_parameters(calibration_df, q)
            if p01 is None or p10 is None:
                p01, p10 = 0.0, 0.0

        matrix = np.kron(
            matrix,
            single_qubit_assignment_matrix(p01, p10)
        )

    return matrix

def counts_to_probability_vector(counts, num_qubits):
    states = [format(i, f"0{num_qubits}b") for i in range(2 ** num_qubits)]
    probabilities = np.array([
        counts.get(state, 0) / max(sum(counts.values()), 1)
        for state in states
    ])
    return states, probabilities

def probability_vector_to_counts(states, probabilities, shots):
    probabilities = np.clip(probabilities, 0.0, None)
    total = probabilities.sum()

    if total <= 0:
        probabilities = np.ones(len(probabilities)) / len(probabilities)
    else:
        probabilities = probabilities / total

    raw_counts = probabilities * shots
    integer_counts = np.floor(raw_counts).astype(int)

    remainder = shots - integer_counts.sum()
    if remainder > 0:
        order = np.argsort(-(raw_counts - integer_counts))
        for idx in order[:remainder]:
            integer_counts[idx] += 1

    return {
        state: int(count)
        for state, count in zip(states, integer_counts)
        if count > 0
    }

def mitigate_readout_counts(counts, num_qubits, calibration_df=None, shots=SHOTS):
    states, measured = counts_to_probability_vector(counts, num_qubits)
    matrix = build_measurement_matrix(num_qubits, calibration_df)

    try:
        mitigated = np.linalg.pinv(matrix) @ measured
    except np.linalg.LinAlgError:
        mitigated = measured

    return probability_vector_to_counts(states, mitigated, shots)

## 5. Load Calibration Snapshot

Readout mitigation requires the calibration snapshot collected in the hardware experiment. If the calibration file is absent, the notebook does not silently substitute invented hardware values.

In [ ]:
calibration_path = PROJECT_ROOT / "data" / "calibration" / f"{BACKEND_NAME}_calibration.csv"

if calibration_path.exists():
    calibration_df = pd.read_csv(calibration_path)
    print("Calibration rows:", len(calibration_df))
else:
    calibration_df = None
    print("Calibration file not found.")
    print("Readout mitigation will use an identity calibration matrix.")

## 6. Apply Readout Mitigation to Available Hardware Results

Only circuits for which raw hardware counts are available are processed.

In [ ]:
readout_counts = {}
readout_rows = []

for name, raw_counts in hardware_counts.items():
    # The raw count keys determine the number of classical output bits.
    num_bits = max(len(state) for state in raw_counts)

    mitigated_counts = mitigate_readout_counts(
        raw_counts,
        num_qubits=num_bits,
        calibration_df=calibration_df,
        shots=sum(raw_counts.values()),
    )

    readout_counts[name] = mitigated_counts

    ideal = ideal_counts.get(name)
    metrics = distribution_metrics(ideal, mitigated_counts) if ideal else {
        "TVD": np.nan,
        "distribution_fidelity": np.nan,
    }

    readout_rows.append({
        "circuit": name,
        "method": "readout_mitigation",
        "shots": sum(raw_counts.values()),
        "TVD": metrics["TVD"],
        "distribution_fidelity": metrics["distribution_fidelity"],
        "benchmark_success_probability": benchmark_success(
            name, mitigated_counts
        ),
    })

readout_df = pd.DataFrame(readout_rows)
readout_df

## 7. Zero-Noise Extrapolation (ZNE) Framework

ZNE requires executions at multiple effective noise levels. A practical hardware implementation can obtain these points through noise scaling / circuit folding.

This notebook provides the analysis framework. The folding function below is intentionally restricted to unitary gates and excludes measurements and barriers.

In [ ]:
from qiskit import QuantumCircuit

def fold_unitary_circuit(qc, scale_factor):
    if scale_factor < 1 or int(scale_factor) != scale_factor or int(scale_factor) % 2 == 0:
        raise ValueError("scale_factor must be an odd integer >= 1.")

    scale_factor = int(scale_factor)
    repetitions = (scale_factor - 1) // 2

    folded = QuantumCircuit(qc.num_qubits, qc.num_clbits)

    for instruction in qc.data:
        operation = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits

        if operation.name in {"measure", "barrier"}:
            folded.append(operation, qargs, cargs)
            continue

        folded.append(operation, qargs, cargs)

        for _ in range(repetitions):
            try:
                inverse = operation.inverse()
            except Exception:
                inverse = None

            if inverse is None:
                raise ValueError(
                    f"Gate {operation.name} cannot be inverted for folding."
                )

            folded.append(inverse, qargs, cargs)
            folded.append(operation, qargs, cargs)

    return folded

## 8. ZNE Extrapolation Utilities

The default analysis uses scale factors **1, 3, and 5** and a linear fit against the effective noise scale. For publication experiments, the actual hardware counts at each scale must be collected and saved before interpreting ZNE improvement.

In [ ]:
def linear_zne(values, scale_factors):
    values = np.asarray(values, dtype=float)
    scales = np.asarray(scale_factors, dtype=float)

    valid = np.isfinite(values)

    if valid.sum() < 2:
        return float("nan")

    coefficients = np.polyfit(scales[valid], values[valid], deg=1)
    return float(np.polyval(coefficients, 0.0))

def zne_success_probability(success_values, scale_factors=(1, 3, 5)):
    return linear_zne(success_values, scale_factors)

## 9. Optional Hardware ZNE Input

The hardware ZNE experiment requires separate measurements at the selected noise scales. If a future execution file named `ibm_kingston_zne_results.csv` exists, it will be loaded automatically.

Expected columns:

`circuit`, `scale_factor`, `shots`, `success_probability`

No fabricated ZNE measurements are generated when the file is absent.

In [ ]:
zne_path = HARDWARE_DIR / f"{BACKEND_NAME}_zne_results.csv"

if zne_path.exists():
    zne_raw_df = pd.read_csv(zne_path)
    print("Loaded ZNE hardware data:", len(zne_raw_df), "rows")
else:
    zne_raw_df = pd.DataFrame()
    print("No hardware ZNE dataset found.")
    print("ZNE results will remain pending until scaled-noise hardware runs are collected.")

In [ ]:
zne_rows = []

if not zne_raw_df.empty:
    for circuit, group in zne_raw_df.groupby("circuit"):
        group = group.sort_values("scale_factor")

        estimate = linear_zne(
            group["success_probability"].values,
            group["scale_factor"].values,
        )

        zne_rows.append({
            "circuit": circuit,
            "method": "ZNE",
            "scale_factors": ",".join(
                str(x) for x in group["scale_factor"].tolist()
            ),
            "zne_success_probability": estimate,
        })

zne_df = pd.DataFrame(zne_rows)
zne_df

## 10. Compare Raw, Readout-Mitigation, and ZNE Results

The comparison table is built only from measurements that actually exist. This avoids treating missing ZNE hardware experiments as zero performance.

In [ ]:
raw_rows = []

for name, counts in hardware_counts.items():
    ideal = ideal_counts.get(name)

    metrics = distribution_metrics(ideal, counts) if ideal else {
        "TVD": np.nan,
        "distribution_fidelity": np.nan,
    }

    raw_rows.append({
        "circuit": name,
        "method": "raw",
        "shots": sum(counts.values()),
        "TVD": metrics["TVD"],
        "distribution_fidelity": metrics["distribution_fidelity"],
        "benchmark_success_probability": benchmark_success(name, counts),
    })

raw_df = pd.DataFrame(raw_rows)

comparison_df = pd.concat(
    [raw_df, readout_df],
    ignore_index=True,
)

if not zne_df.empty:
    comparison_df = comparison_df.merge(
        zne_df[["circuit", "zne_success_probability"]],
        on="circuit",
        how="left",
    )
else:
    comparison_df["zne_success_probability"] = np.nan

comparison_df

## 11. Calculate Improvement Metrics

Improvement is reported relative to the raw hardware result. Positive fidelity improvement and reduced TVD indicate movement toward the ideal reference distribution.

In [ ]:
raw_reference = (
    raw_df
    .set_index("circuit")
    [["TVD", "distribution_fidelity", "benchmark_success_probability"]]
    .rename(columns=lambda x: f"raw_{x}")
)

analysis_df = comparison_df.merge(
    raw_reference,
    left_on="circuit",
    right_index=True,
    how="left",
)

analysis_df["TVD_reduction_vs_raw"] = (
    analysis_df["raw_TVD"] - analysis_df["TVD"]
)

analysis_df["fidelity_improvement_vs_raw"] = (
    analysis_df["distribution_fidelity"]
    - analysis_df["raw_distribution_fidelity"]
)

analysis_df["success_improvement_vs_raw"] = (
    analysis_df["benchmark_success_probability"]
    - analysis_df["raw_benchmark_success_probability"]
)

analysis_df

## 12. Mitigation Overhead

QEM is not evaluated only by accuracy. The study also records execution overhead because mitigation can require additional circuit executions, calibration data, or classical post-processing.

For the current analysis:

- Raw = one hardware execution.
- Readout mitigation = one circuit execution plus calibration overhead.
- ZNE = number of noise-scale executions.

In [ ]:
overhead_rows = []

for circuit in sorted(hardware_counts):
    overhead_rows.append({
        "circuit": circuit,
        "raw_execution_multiplier": 1,
        "readout_execution_multiplier": 1,
        "zne_execution_multiplier": (
            int(zne_raw_df[
                zne_raw_df["circuit"] == circuit
            ]["scale_factor"].count())
            if not zne_raw_df.empty else np.nan
        ),
    })

overhead_df = pd.DataFrame(overhead_rows)
overhead_df

## 13. Method-Level Summary

This table summarizes the measured distribution fidelity and TVD by mitigation method. It should be interpreted together with execution overhead rather than as an isolated ranking.

In [ ]:
method_summary = (
    comparison_df
    .groupby("method", dropna=False)
    .agg(
        circuits=("circuit", "nunique"),
        mean_TVD=("TVD", "mean"),
        mean_distribution_fidelity=("distribution_fidelity", "mean"),
        mean_success_probability=("benchmark_success_probability", "mean"),
    )
    .reset_index()
)

method_summary

## 14. Save QEM Analysis Data

In [ ]:
raw_path = MITIGATED_DIR / f"{BACKEND_NAME}_raw_metrics.csv"
readout_path = MITIGATED_DIR / f"{BACKEND_NAME}_readout_mitigation.csv"
comparison_path = MITIGATED_DIR / f"{BACKEND_NAME}_qem_comparison.csv"
overhead_path = MITIGATED_DIR / f"{BACKEND_NAME}_qem_overhead.csv"
summary_path = MITIGATED_DIR / f"{BACKEND_NAME}_qem_method_summary.csv"

raw_df.to_csv(raw_path, index=False)
readout_df.to_csv(readout_path, index=False)
analysis_df.to_csv(comparison_path, index=False)
overhead_df.to_csv(overhead_path, index=False)
method_summary.to_csv(summary_path, index=False)

print("Saved:")
for path in [raw_path, readout_path, comparison_path, overhead_path, summary_path]:
    print("-", path)

## 15. Publication-Oriented Checks

Before reporting QEM improvement in the paper, verify:

1. Raw and mitigated runs use the same benchmark definition.
2. Hardware calibration data correspond to the same experimental period.
3. ZNE uses multiple measured noise scales rather than simulated substitute values.
4. Shot counts and job IDs are preserved.
5. Statistical uncertainty is reported for measured probabilities.
6. Mitigation overhead is reported together with accuracy improvement.

In [ ]:
print("QEM analysis stage completed.")
print("Raw hardware circuits:", len(raw_df))
print("Readout-mitigated circuits:", len(readout_df))
print("ZNE hardware circuits:", len(zne_df))
print("\nNext notebook: 06_final_results.ipynb")

## Next Step

**`06_final_results.ipynb`** will assemble the complete experimental evidence: ideal vs noisy vs IBM hardware, raw vs mitigated performance, circuit/hardware characteristics, statistical analysis, publication-quality tables, and figures for the paper.